# Week 12 Optional: Automated Evaluation Metrics (BLEU, ROUGE & Beyond)

## Overview

This is a **supplementary notebook** for advanced learners who want to explore
automated text evaluation metrics beyond the manual accuracy/format checks
we built in the main notebook.

## What You'll Learn

1. **BLEU score** — measures n-gram overlap between generated and reference text
2. **ROUGE score** — measures recall-oriented overlap (common in summarization)
3. **When to use which** — matching the right metric to the right task
4. **Limitations** — why automated metrics can be misleading for LLM evaluation

## Prerequisites

- Completed Week 12 main notebook (especially Section 2: LLM Evaluation)
- Familiarity with the fraud classification task and Flan-T5 outputs

## Why This Matters

In the main notebook, we evaluated LLM outputs by checking if the label was
correct (exact match). But what about **free-text outputs** like explanations,
summaries, or reasoning? You can't do exact match on those — you need metrics
that measure "how similar" two texts are. That's where BLEU and ROUGE come in.

```
Exact Match (main notebook):     "fraud" == "fraud"  → 1.0
Text Similarity (this notebook): "unauthorized transfer" ≈ "suspicious wire transfer" → 0.65
```

# Section 0: Environment Setup

In [ ]:
# =============================================================================
# INSTALL REQUIRED LIBRARIES
# =============================================================================
# evaluate: HuggingFace's unified evaluation library
# rouge_score: Backend for ROUGE metric
# nltk: Backend for BLEU metric (tokenization)

!pip install -q evaluate rouge_score nltk transformers torch accelerate

import evaluate
import nltk
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import time

# Download NLTK tokenizer data (needed for BLEU)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print(f"evaluate version: {evaluate.__version__}")
print("\n✅ Environment ready!")

In [ ]:
# =============================================================================
# SAMPLE DATA — Generated vs Reference Explanations
# =============================================================================
# We'll use fraud reasoning explanations as our evaluation target.
# "references" = what a human analyst would write
# "predictions" = what the LLM generated

references = [
    "This is fraud because the wire transfer was unauthorized and sent to an unknown overseas account at an unusual hour.",
    "This is legitimate because it matches the customer's 18-month Netflix subscription history.",
    "This is fraud because multiple ATM withdrawals occurred in different cities within 2 hours and the card was reported lost.",
    "This is legitimate because the Amazon purchase ships to the address on file and matches purchase history.",
    "This is fraud because the customer was in Chicago but the purchase was made in Miami with no travel alert.",
    "This is legitimate because the payroll deposit matches the bi-weekly schedule and employment records.",
    "This is fraud because multiple small purchases at unknown merchants from different IP addresses suggest card testing.",
    "This is legitimate because the Whole Foods purchase is consistent with the customer's weekly shopping pattern.",
]

# Simulated LLM outputs (varying quality to show metric differences)
predictions = [
    "Fraud detected due to unauthorized wire transfer to overseas account at 3:47 AM with no international history.",
    "Legitimate transaction consistent with subscription history for Netflix streaming service.",
    "This appears to be fraud with ATM withdrawals in multiple cities in a short time period.",
    "Legitimate purchase at Amazon.com shipping to known address.",
    "Fraud because purchase location does not match customer location.",
    "Legitimate payroll deposit from employer matching records.",
    "The transaction involves small purchases at various digital stores which is suspicious activity.",
    "Legitimate grocery purchase at regular store.",
]

print(f"Loaded {len(references)} reference/prediction pairs")
print(f"\nExample:")
print(f"  Reference:  {references[0][:80]}...")
print(f"  Prediction: {predictions[0][:80]}...")

# Section 1: BLEU Score

**BLEU** (Bilingual Evaluation Understudy) was originally designed for machine
translation, but it's widely used to evaluate any generated text against a reference.

## How BLEU Works

BLEU measures **precision** of n-grams: what fraction of the n-grams in the
generated text also appear in the reference?

```
Reference:  "the cat sat on the mat"
Prediction: "the cat on the mat"

1-gram (unigram) precision: 5/5 = 1.0  (all words in prediction appear in reference)
2-gram (bigram) precision:  3/4 = 0.75 ("the cat", "on the", "the mat" match; "cat on" doesn't)
```

BLEU combines 1-gram through 4-gram precision with a **brevity penalty**
(penalizes very short outputs that would otherwise get perfect precision).

## When to Use BLEU

- Comparing generated text to a reference translation/answer
- Good for: translation, structured outputs, short answers
- **Not ideal for**: long free-form text, creative writing, or when many valid answers exist

In [ ]:
# =============================================================================
# DEMO: Computing BLEU with HuggingFace evaluate
# =============================================================================

bleu_metric = evaluate.load("bleu")

# BLEU expects references as list-of-lists (each prediction can have multiple references)
bleu_result = bleu_metric.compute(
    predictions=predictions,
    references=references
)

print("BLEU Score (corpus-level):")
print(f"  BLEU:          {bleu_result['bleu']:.4f}")
print(f"  Precision-1:   {bleu_result['precisions'][0]:.4f}")
print(f"  Precision-2:   {bleu_result['precisions'][1]:.4f}")
print(f"  Precision-3:   {bleu_result['precisions'][2]:.4f}")
print(f"  Precision-4:   {bleu_result['precisions'][3]:.4f}")
print(f"  Brevity pen.:  {bleu_result['brevity_penalty']:.4f}")
print(f"  Ref length:    {bleu_result['reference_length']}")
print(f"  Pred length:   {bleu_result['translation_length']}")

print(f"\n💡 BLEU ranges 0-1. Scores above 0.3 are decent for free-text generation.")
print(f"   The brevity penalty is < 1.0 when predictions are shorter than references.")

In [ ]:
# =============================================================================
# DEMO: Per-Sample BLEU — See Which Explanations Score Best/Worst
# =============================================================================

per_sample_bleu = []
for i, (pred, ref) in enumerate(zip(predictions, references)):
    score = bleu_metric.compute(predictions=[pred], references=[ref])
    per_sample_bleu.append({
        'idx': i,
        'bleu': score['bleu'],
        'brevity_penalty': score['brevity_penalty'],
        'pred_preview': pred[:50] + "...",
        'ref_preview': ref[:50] + "...",
    })

bleu_df = pd.DataFrame(per_sample_bleu)
display(bleu_df[['idx', 'bleu', 'brevity_penalty', 'pred_preview']])

best = bleu_df.loc[bleu_df['bleu'].idxmax()]
worst = bleu_df.loc[bleu_df['bleu'].idxmin()]
print(f"\nBest BLEU ({best['bleu']:.3f}):  {predictions[int(best['idx'])]}")
print(f"Worst BLEU ({worst['bleu']:.3f}): {predictions[int(worst['idx'])]}")
print(f"\n💡 Notice: shorter predictions get penalized by the brevity penalty.")

# Section 2: ROUGE Score

**ROUGE** (Recall-Oriented Understudy for Gisting Evaluation) was designed
for summarization. Unlike BLEU (which measures precision), ROUGE focuses
on **recall**: how much of the reference text is captured in the prediction?

## ROUGE Variants

| Variant | What It Measures |
|---------|-----------------|
| **ROUGE-1** | Unigram (single word) overlap |
| **ROUGE-2** | Bigram (two-word) overlap |
| **ROUGE-L** | Longest Common Subsequence — captures sentence-level structure |

Each variant reports **precision**, **recall**, and **F1**:
- **Precision**: What fraction of prediction n-grams appear in the reference?
- **Recall**: What fraction of reference n-grams appear in the prediction?
- **F1**: Harmonic mean of precision and recall

## When to Use ROUGE

- Summarization tasks (is the summary capturing key information?)
- Good for: checking if important content is preserved
- **Not ideal for**: exact wording matters, or when output format is critical

In [ ]:
# =============================================================================
# DEMO: Computing ROUGE with HuggingFace evaluate
# =============================================================================

rouge_metric = evaluate.load("rouge")

rouge_result = rouge_metric.compute(
    predictions=predictions,
    references=references
)

print("ROUGE Scores (corpus-level):")
print(f"  ROUGE-1: {rouge_result['rouge1']:.4f}")
print(f"  ROUGE-2: {rouge_result['rouge2']:.4f}")
print(f"  ROUGE-L: {rouge_result['rougeL']:.4f}")

print(f"\n💡 ROUGE-1 measures word overlap (are the right words present?).")
print(f"   ROUGE-2 measures phrase overlap (are phrases preserved?).")
print(f"   ROUGE-L measures structural similarity (is the sentence order similar?).")

In [ ]:
# =============================================================================
# DEMO: Per-Sample ROUGE — Compare Individual Explanations
# =============================================================================

per_sample_rouge = []
for i, (pred, ref) in enumerate(zip(predictions, references)):
    score = rouge_metric.compute(predictions=[pred], references=[ref])
    per_sample_rouge.append({
        'idx': i,
        'rouge1': score['rouge1'],
        'rouge2': score['rouge2'],
        'rougeL': score['rougeL'],
        'pred_preview': pred[:50] + "...",
    })

rouge_df = pd.DataFrame(per_sample_rouge)
display(rouge_df)

print(f"\n💡 Compare these to the BLEU scores above — same samples, different metrics.")
print(f"   ROUGE tends to be more forgiving because it focuses on recall (coverage).")
print(f"   BLEU is stricter because it focuses on precision (exactness).")

# Section 3: BLEU vs ROUGE — When Each Metric Fails

Automated metrics have known limitations. Understanding when they fail
is just as important as knowing how to compute them.

## Known Limitations

| Problem | Example | Which Metric Fails |
|---------|---------|-------------------|
| **Synonyms** | "unauthorized" vs "illegal" — same meaning, zero n-gram match | Both |
| **Paraphrasing** | "the cat sat on the mat" vs "on the mat, the cat was sitting" | BLEU more than ROUGE-L |
| **Hallucination** | Generated text adds plausible but wrong facts | Neither detects this! |
| **Correct but different** | Two valid explanations with no word overlap | Both give low scores |

## The Key Insight

> **Automated metrics measure text similarity, not semantic correctness.**
> A high BLEU/ROUGE score means the text uses similar words — not that it's right.
> A low score doesn't mean the output is wrong — just that it's worded differently.

In [ ]:
# =============================================================================
# DEMO: When Metrics Fail — Illustrative Examples
# =============================================================================

# These pairs show when BLEU/ROUGE give misleading scores
failure_cases = [
    {
        "name": "Synonym (correct but low score)",
        "ref":  "The transaction is fraudulent due to unauthorized access.",
        "pred": "The purchase is illegal because of unapproved entry.",
    },
    {
        "name": "Hallucination (wrong but high score)",
        "ref":  "This is fraud because of the unusual overseas transfer.",
        "pred": "This is fraud because of the unusual overseas transfer and the customer's criminal record.",
    },
    {
        "name": "Word salad (wrong but moderate score)",
        "ref":  "Legitimate purchase matching customer history at local store.",
        "pred": "Customer history local store matching purchase legitimate at.",
    },
]

print("Failure Cases — When Automated Metrics Are Misleading:")
print("=" * 60)
for case in failure_cases:
    bleu_s = bleu_metric.compute(predictions=[case['pred']], references=[case['ref']])
    rouge_s = rouge_metric.compute(predictions=[case['pred']], references=[case['ref']])
    print(f"\n{case['name']}:")
    print(f"  Reference:  {case['ref']}")
    print(f"  Prediction: {case['pred']}")
    print(f"  BLEU: {bleu_s['bleu']:.3f}  |  ROUGE-1: {rouge_s['rouge1']:.3f}  |  ROUGE-L: {rouge_s['rougeL']:.3f}")

print(f"\n💡 The hallucination case gets a HIGH score because it contains all the")
print(f"   reference words plus extra (made-up) information. Metrics can't detect this!")

## Lab: Build a Combined Evaluation Report

### Your Task

Combine BLEU, ROUGE, and your accuracy metric from the main notebook into a
single evaluation function. Run it on all 8 prediction/reference pairs and
produce a comprehensive report.

### Steps

1. **Write `full_evaluation(prediction, reference, actual_label, predicted_label)`**
   that returns a dict with: `bleu`, `rouge1`, `rouge2`, `rougeL`, `accuracy`,
   and a combined `quality_score` (weighted average of your choice)
2. **Run on all 8 pairs** and create a scores DataFrame
3. **Identify the best and worst outputs** by your combined score
4. **Create a visualization** comparing BLEU vs ROUGE-L for each sample
5. **Answer**: Do BLEU and ROUGE agree on which outputs are best/worst?

### Expected Output

- DataFrame with all metrics per sample
- Scatter plot: BLEU (x) vs ROUGE-L (y) with sample labels
- Print: correlation between BLEU and ROUGE-L

In [ ]:
# =============================================================================
# LAB: BUILD A COMBINED EVALUATION REPORT
# =============================================================================

# Ground truth labels for our 8 transactions
actual_labels = ["fraud", "legitimate", "fraud", "legitimate",
                 "fraud", "legitimate", "fraud", "legitimate"]
# Simulated predicted labels (from the LLM)
predicted_labels = ["fraud", "legitimate", "fraud", "legitimate",
                    "fraud", "legitimate", "fraud", "legitimate"]


def full_evaluation(prediction, reference, actual_label, predicted_label):
    """
    Compute BLEU, ROUGE, accuracy, and a combined quality score.
    """
    # YOUR CODE: Compute BLEU score for this pair
    bleu_score = None  # YOUR CODE

    # YOUR CODE: Compute ROUGE scores for this pair
    rouge1_score = None  # YOUR CODE
    rouge2_score = None  # YOUR CODE
    rougeL_score = None  # YOUR CODE

    # YOUR CODE: Compute accuracy (1 if labels match, 0 if not)
    acc = None  # YOUR CODE

    # YOUR CODE: Compute combined quality_score
    # Suggestion: 0.4 * accuracy + 0.2 * bleu + 0.2 * rouge1 + 0.2 * rougeL
    quality_score = None  # YOUR CODE

    return {
        'bleu': bleu_score,
        'rouge1': rouge1_score,
        'rouge2': rouge2_score,
        'rougeL': rougeL_score,
        'accuracy': acc,
        'quality_score': quality_score,
    }


# YOUR CODE: Run on all 8 pairs
all_scores = []  # YOUR CODE

# YOUR CODE: Create DataFrame
combined_df = None  # YOUR CODE

# YOUR CODE: Scatter plot — BLEU (x) vs ROUGE-L (y)

# YOUR CODE: Print correlation between BLEU and ROUGE-L

# Verification
if combined_df is not None:
    display(combined_df)
    print(f"\n🎉 Lab complete!")
else:
    print("❌ Lab incomplete — fill in the YOUR CODE sections above")

# Summary: Choosing the Right Evaluation Metric

| Metric | Best For | Measures | Watch Out For |
|--------|----------|----------|---------------|
| **Exact Match** | Classification labels | Correctness | Misses partial credit |
| **BLEU** | Translation, structured output | Precision (n-gram) | Penalizes paraphrasing |
| **ROUGE** | Summarization, explanations | Recall (n-gram) | Can't detect hallucination |
| **Human Eval** | Everything (gold standard) | True quality | Expensive, slow, subjective |

## The Practical Takeaway

For most LLM evaluation in production, you'll use a **combination**:
1. **Exact match** for structured fields (labels, categories)
2. **ROUGE-L** for free-text quality (reasoning, explanations)
3. **Human review** on a sample for final validation
4. **LLM-as-judge** (Week 13+) — using a stronger model to evaluate a weaker one

## Resources

- [HuggingFace Evaluate Docs](https://huggingface.co/docs/evaluate/)
- [BLEU Paper](https://aclanthology.org/P02-1040/) — Papineni et al., 2002
- [ROUGE Paper](https://aclanthology.org/W04-1013/) — Lin, 2004
- [BERTScore](https://github.com/Tiiiger/bert_score) — semantic similarity using embeddings (beyond this notebook)